# Model Validation Diagnostics

This notebook validates the final latent potential outputs for Smil Labs. It checks submission shape, uplift behavior, segment summaries, POI/catchment relationships, and top-risk predictions.

In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'Results'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
DOCS_DIR = PROJECT_ROOT / 'Docs'

PLATFORM_FILE = RESULTS_DIR / 'smil_labs_predictions.csv'
FULL_FILE = RESULTS_DIR / 'smil_labs_predictions_full_20000.csv'
DIAGNOSTICS_FILE = GOLD_DIR / 'prediction_diagnostics.csv'

platform = pd.read_csv(PLATFORM_FILE)
full = pd.read_csv(FULL_FILE)
diagnostics = pd.read_csv(DIAGNOSTICS_FILE)

print(platform.shape, full.shape, diagnostics.shape)

## Schema Checks

In [ ]:
schema_checks = pd.DataFrame([
    {'check': 'platform_rows', 'value': len(platform), 'expected': 914, 'passed': len(platform) == 914},
    {'check': 'platform_columns', 'value': ', '.join(platform.columns), 'expected': 'row_id, Maximum_Monthly_Liters', 'passed': list(platform.columns) == ['row_id', 'Maximum_Monthly_Liters']},
    {'check': 'platform_missing_values', 'value': int(platform.isna().sum().sum()), 'expected': 0, 'passed': int(platform.isna().sum().sum()) == 0},
    {'check': 'platform_unique_row_id', 'value': platform['row_id'].nunique(), 'expected': len(platform), 'passed': platform['row_id'].nunique() == len(platform)},
    {'check': 'full_rows', 'value': len(full), 'expected': 20000, 'passed': len(full) == 20000},
    {'check': 'full_missing_values', 'value': int(full.isna().sum().sum()), 'expected': 0, 'passed': int(full.isna().sum().sum()) == 0},
])
schema_checks

## Prediction and Uplift Distribution

In [ ]:
distribution = diagnostics[
    ['observed_max_monthly_liters', 'Maximum_Monthly_Liters', 'uplift_ratio_vs_max', 'constraint_score', 'catchment_density_score', 'poi_demand_score']
].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).round(3)
distribution

## Segment Validation

In [ ]:
by_size = diagnostics.groupby('Outlet_Size').agg(
    outlets=('Outlet_ID', 'count'),
    avg_potential=('Maximum_Monthly_Liters', 'mean'),
    median_potential=('Maximum_Monthly_Liters', 'median'),
    avg_uplift=('uplift_ratio_vs_max', 'mean'),
    avg_constraint=('constraint_score', 'mean'),
    avg_poi_score=('poi_demand_score', 'mean'),
).round(3).sort_values('avg_potential', ascending=False)

by_type = diagnostics.groupby('Outlet_Type').agg(
    outlets=('Outlet_ID', 'count'),
    avg_potential=('Maximum_Monthly_Liters', 'mean'),
    median_potential=('Maximum_Monthly_Liters', 'median'),
    avg_uplift=('uplift_ratio_vs_max', 'mean'),
    avg_constraint=('constraint_score', 'mean'),
    avg_poi_score=('poi_demand_score', 'mean'),
).round(3).sort_values('avg_potential', ascending=False)

by_distributor = diagnostics.groupby('Distributor_ID').agg(
    outlets=('Outlet_ID', 'count'),
    avg_potential=('Maximum_Monthly_Liters', 'mean'),
    avg_uplift=('uplift_ratio_vs_max', 'mean'),
    avg_constraint=('constraint_score', 'mean'),
).round(3).sort_values('avg_potential', ascending=False)

display(by_size)
display(by_type)
display(by_distributor)

## Top Potential and Top Uplift Reviews

In [ ]:
review_columns = [
    'Outlet_ID', 'Outlet_Size', 'Outlet_Type', 'Distributor_ID',
    'observed_max_monthly_liters', 'Maximum_Monthly_Liters', 'uplift_ratio_vs_max',
    'constraint_score', 'catchment_density_score', 'poi_demand_score',
]

top_potential = diagnostics.sort_values('Maximum_Monthly_Liters', ascending=False).head(100)[review_columns]
top_uplift = diagnostics.sort_values('uplift_ratio_vs_max', ascending=False).head(100)[review_columns]

top_potential.to_csv(GOLD_DIR / 'validation_top_100_potential.csv', index=False)
top_uplift.to_csv(GOLD_DIR / 'validation_top_100_uplift.csv', index=False)

display(top_potential.head(10).round(3))
display(top_uplift.head(10).round(3))

## Signal Correlation

In [ ]:
correlation = diagnostics[
    ['poi_demand_score', 'catchment_density_score', 'constraint_score', 'Maximum_Monthly_Liters', 'uplift_ratio_vs_max']
].corr().round(3)
correlation

## Write Validation Summary

In [ ]:
def md_table(df: pd.DataFrame) -> str:
    display_df = df.reset_index() if df.index.name is not None else df.copy()
    display_df = display_df.where(pd.notna(display_df), '')
    lines = ['| ' + ' | '.join(map(str, display_df.columns)) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(display_df.columns)) + ' |')
    for row in display_df.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(map(str, row)) + ' |')
    return '\n'.join(lines)

lines = [
    '# Model Validation Summary',
    '',
    'This validation summary was generated from `Notebooks/04_model_validation.ipynb`.',
    '',
    '## Schema Checks',
    '',
    md_table(schema_checks),
    '',
    '## Key Distribution Metrics',
    '',
    md_table(distribution.loc[['mean', '50%', '90%', '95%', '99%', 'max']].reset_index()),
    '',
    '## Outlet Size Summary',
    '',
    md_table(by_size.reset_index()),
    '',
    '## Outlet Type Summary',
    '',
    md_table(by_type.reset_index()),
    '',
    '## Signal Correlation',
    '',
    md_table(correlation.reset_index()),
    '',
    '## Review Artifacts',
    '',
    '- `data/gold/validation_top_100_potential.csv`',
    '- `data/gold/validation_top_100_uplift.csv`',
]

(DOCS_DIR / 'model_validation_summary.md').write_text('\n'.join(lines), encoding='utf-8')
print(DOCS_DIR / 'model_validation_summary.md')